## Notebook to learn to play with tif images

In [1]:
import sys
import random
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt

import experiment_settings
import build_model
import train_model
import build_data

import tensorflow as tf

In [2]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on
# print(tf.config.list_physical_devices('GPU'))

python version = 3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:12:31) [Clang 14.0.6 ]
numpy version = 1.23.2
tensorflow version = 2.10.0


In [3]:
# GET SETTINGS
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)

# SET RANDOM SEEDS
np.random.seed(settings["rng_seed"])
random.seed(settings["rng_seed"])
tf.random.set_seed(settings["rng_seed"])

In [4]:
# LOAD THE DATA
imp.reload(build_data)

(tagyear_train, 
 taglat_train, 
 taglon_train,
 tagyear_val, 
 taglat_val, 
 taglon_val, 
 ) = build_data.make_sample_list(settings)

tfds_train = build_data.build_tf_dataset(settings, tagyear_train, taglat_train, taglon_train, settings["batch_size"])
tfds_val = build_data.build_tf_dataset(settings, tagyear_val, taglat_val, taglon_val, settings["batch_size"])

batch_shape = np.shape(next(tfds_val.as_numpy_iterator())[0])
print(f"{batch_shape = }")

output region shape = (90, 90)
ntrain = 6400, nval = 800
Metal device set to: Apple M1 Max

systemMemory: 64.00 GB
maxCacheSize: 24.00 GB

batch_shape = (32, 114, 114, 6)


In [5]:
# TRAIN THE MODEL
imp.reload(build_model)
imp.reload(train_model)

tf.keras.backend.clear_session()
model = build_model.build_model(settings, input_shape=batch_shape[1:])

model, fit_summary, history, settings = train_model.train_model(settings, model, tfds_train, tfds_val)

fit_summary

Epoch 1/1000
200/200 [==============================] - 40s 198ms/step - loss: 0.0242 - mae: 0.1145 - val_loss: 0.0172 - val_mae: 0.0944 - lr: 0.0010
Epoch 2/1000
200/200 [==============================] - 40s 198ms/step - loss: 0.0155 - mae: 0.0931 - val_loss: 0.0144 - val_mae: 0.0860 - lr: 0.0010
Epoch 3/1000
200/200 [==============================] - 40s 199ms/step - loss: 0.0134 - mae: 0.0869 - val_loss: 0.0137 - val_mae: 0.0756 - lr: 0.0010
Epoch 4/1000
200/200 [==============================] - 40s 200ms/step - loss: 0.0128 - mae: 0.0845 - val_loss: 0.0163 - val_mae: 0.0926 - lr: 0.0010
Epoch 5/1000
179/200 [=========================>....] - ETA: 3s - loss: 0.0121 - mae: 0.0823